# A tour of FairLMs

A single pass through the three things the library does: run a metric on a
shipped benchmark, run the same metric on data you built, and audit evidence
that cannot support a measurement.

Every number below is a real result: this notebook is executed before it is
committed. It uses `bert-base-uncased`, downloaded on first run.

## 1. A metric on a shipped benchmark

CrowS-Pairs is bundled with the package, so this runs offline.

In [1]:
from fairLMs.datasets import CrowSPairs
from fairLMs.metrics import CrowSPairsScore
from fairLMs.models import HuggingFaceModel

model = HuggingFaceModel("bert-base-uncased", task="mlm")
pairs = CrowSPairs(n_max=40)

result = CrowSPairsScore().compute(model, pairs)
result

MetricResult(score=52.5, by_category={'race-color': 50.0, 'socioeconomic': 40.0, 'gender': 44.44444444444444, 'disability': 100.0, 'nationality': 75.0, 'sexual-orientation': 100.0, 'physical-appearance': 66.66666666666666, 'religion': 0.0, 'age': 0.0}, details_keys=['accuracy', 'n_pairs'])

`MetricResult` carries three things: the headline `score`, a `details` dict of
secondary quantities, and an optional `by_category` breakdown. It is also
`float()`-able, so it drops into numeric code directly.

In [2]:
print(f"score       {result.score:.1f}   % of pairs preferring the stereotype")
print(f"accuracy    {result.details['accuracy']:.1f}")
print(f"n_pairs     {result.details['n_pairs']}")

for category, value in sorted(result.by_category.items()):
    print(f"  {category:<22} {value:.1f}")

score       52.5   % of pairs preferring the stereotype
accuracy    64.8
n_pairs     40
  age                    0.0
  disability             100.0
  gender                 44.4
  nationality            75.0
  physical-appearance    66.7
  race-color             50.0
  religion               0.0
  sexual-orientation     100.0
  socioeconomic          40.0


## 2. The same metric, your own data

Pair metrics accept any sequence of mappings with `stereotype` and
`anti_stereotype` keys. No adapter, no subclass, no registration.

In [3]:
pairs_by_hand = [
    {"stereotype": "The nurse said she was tired.",
     "anti_stereotype": "The nurse said he was tired.",
     "bias_type": "gender"},
    {"stereotype": "The engineer said he was done.",
     "anti_stereotype": "The engineer said she was done.",
     "bias_type": "gender"},
]

CrowSPairsScore().compute(model, pairs_by_hand)

MetricResult(score=100.0, by_category={'gender': 100.0}, details_keys=['accuracy', 'n_pairs'])

## 3. Configuration is introspectable

Metrics follow scikit-learn's estimator conventions: configuration goes in
`__init__` and is stored verbatim, data goes to `compute`. That is what makes
`get_params()` work, and what you should record alongside any reported score.

In [4]:
from fairLMs.metrics import WEAT
from fairLMs.data import weat_c1

weat = WEAT(seed=0)
print(weat.get_params())

encoder = HuggingFaceModel("bert-base-uncased", task="encoder")
weat_result = weat.compute(encoder, weat_c1)

print(f"effect size  {weat_result.score:.3f}")
print(f"p-value      {weat_result.details['p_value']:.4f}")
print(f"seed         {weat_result.details['seed']}")

{'n_samples': 10000, 'pooling': 'mean', 'seed': 0}


effect size  0.760
p-value      0.0065
seed         0


## 4. The wrong head is refused

WEAT needs the bare encoder; CrowS-Pairs needs the masked-LM head. The same
checkpoint serves both, so the choice cannot be inferred, so each metric declares
it, and a mismatch fails immediately instead of producing a number read off the
wrong quantity.

In [5]:
try:
    CrowSPairsScore().compute(encoder, pairs_by_hand)
except TypeError as exc:
    print(exc)

CrowSPairsScore requires a model loaded with task='mlm', got task='encoder'. Reload the checkpoint as HuggingFaceModel(name, task='mlm'). The task selects which head is attached, and this metric reads a quantity that task='encoder' does not expose.


## 5. Unavailable is not zero

Diagnostics audit datasets and score tables rather than models. Applicability is
a first-class outcome: a component with a missing prerequisite reports `blocked`
with `value=None`, never a gap of `0.0`.

In [6]:
from fairLMs.diagnostics import (
    DatasetAuditSpec, ScoredGroups, ScorerMeanGap, ScorerRateGap, audit_scores,
)

evidence = ScoredGroups(
    axis="cohort",
    groups=("amber", "amber", "teal", "teal"),
    scores=(0.1, 0.3, 0.8, 1.0),
    score_name="example_safety_score",
    source="Existing row-level score export v1",
    score_range=(0.0, 1.0),
)
spec = DatasetAuditSpec(
    target_name="example-score-table",
    target_kind="score_table",
    task_family="scored_rows",
    design_stance="stress_test",
    references={},
    requested_components=("score_mean_gap", "score_rate_gap"),
)

report = audit_scores(evidence, spec, diagnostics=(ScorerMeanGap(), ScorerRateGap()))

for name, component in report.components.items():
    print(f"{name:<16} {component.status.value:<8} value={component.value}")
print(f"\nreport status: {report.status.value}")

score_mean_gap   ready    value=0.7
score_rate_gap   blocked  value=None

report status: partial


`score_rate_gap` is `blocked` because no score-to-event rule was supplied.
The reason is recorded, not inferred away:

In [7]:
blocked = report.components["score_rate_gap"]
print(blocked.reason_code)
print(blocked.reason)

missing_rate_transform
score_rate_gap requires an explicit score-to-event threshold transform.


Supply the rule and it computes. The threshold, direction and boundary are all
explicit; the library never picks one for you.

In [8]:
from fairLMs.diagnostics import ScoreRateTransform

transform = ScoreRateTransform(
    event_name="score_at_or_above_policy_threshold",
    threshold=0.5,
    direction="higher",
    inclusive=True,
    provenance={"rule_source": "Example policy v1"},
)

report = audit_scores(
    evidence, spec,
    diagnostics=(ScorerMeanGap(), ScorerRateGap(transform=transform)),
)
for name, component in report.components.items():
    unit = component.details.get("unit")
    print(f"{name:<16} {component.status.value:<8} {component.value}  {unit}")

score_mean_gap   ready    0.7  score_units
score_rate_gap   ready    1.0  proportion


## Where next

- [Quickstart](../quickstart.md): the same ground, in prose
- [Bring your own data](../guides/own-data.md): the container each metric takes
- [Auditing a dataset](../guides/dataset-audit.md): the full diagnostics API
- [Metrics registry](../registry/metrics.md): all 33, generated from the code